# 🎓 Comprehensive Guide to Logistic Regression and Log-Loss
**Course:** CSCN8010 - Foundations of Machine Learning  
**Group 2:** Ali Cihan Ozdemir (Driver) & Lohith (Navigator)  
*Participation Note: Roshan did not participate in this assignment.*

---
This notebook is an educational walkthrough designed to explain the theory, mathematics, and implementation of Statistical Classification using Logistic Regression and the Log-Loss function (Binary Cross-Entropy).


## 1. Introduction to Logistic Regression

**Logistic Regression** is a statistical method used for binary classification problems—where the goal is to assign each input to one of two possible categories (e.g., Spam vs. Not Spam, Pass vs. Fail). 

Unlike linear regression, which predicts continuous unbounded values, logistic regression predicts the **probability** that an input belongs to a particular class. Because probabilities must strictly lie between 0 and 1, we cannot simply use a straight line. We need a function that maps any real number into the (0, 1) range.


### The Logistic (Sigmoid) Function

At the heart of logistic regression is the **sigmoid function**. It takes any real-valued number and maps it into a value between 0 and 1, creating an "S-shaped" curve. 

Mathematically:
$$ \sigma(z) = \frac{1}{1 + e^{-z}} $$
where $z = wx + b$ (our linear model).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the highly-important Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Generate dummy data for the x-axis
z = np.linspace(-10, 10, 100)
p = sigmoid(z)

# Plot the Sigmoid Curve
plt.figure(figsize=(10, 5))
plt.style.use('dark_background') # Using dark style for better contract
plt.plot(z, p, color='cyan', linewidth=3, label='Sigmoid Curve $\sigma(z)$')
plt.axhline(0.5, color='gray', linestyle='--', label='Decision Boundary (0.5)')
plt.axvline(0, color='gray', linestyle=':')
plt.title("The Standard Sigmoid Function")
plt.xlabel("z (Linear Combination)")
plt.ylabel("Probability p(z)")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 2. Statistical Classification & The Decision Boundary

**Classification** is the process of taking an input and assigning it to a specific category.

Once the logistic regression model learns the probabilities using the sigmoid curve, we apply a **decision threshold** to make a final discrete classification. 

> **Rule:** If the predicted probability $P(y=1|X)$ is $\ge 0.5$, we classify the sample as Class 1. Otherwise, we classify it as Class 0.

Let's visualize how a decision boundary separates two classes in a 2D space.


In [ ]:
from sklearn.datasets import make_blobs

# Generate linearly separable test data
X_blobs, y_blobs = make_blobs(n_samples=100, centers=[(5, 5), (0, 0)], n_features=2, random_state=42, cluster_std=1.5)

# A manual decision boundary for illustration
x_decision = np.linspace(-3, 8, 100)
y_decision = -1.2 * x_decision + 5.5 

plt.figure(figsize=(10, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='bwr', edgecolor='k', s=60, label="Data Points")
plt.plot(x_decision, y_decision, 'y--', linewidth=2, label="Decision Boundary")
plt.title("Separating Classes with a Decision Boundary")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 3. The Log-Loss Function (Cross Entropy)

To train a Logistic Regression model, we need a loss function. We cannot use Mean Squared Error (MSE) because combining it with the Sigmoid function creates a non-convex loss surface (with local minima where Gradient Descent can get stuck).

Instead, we use **Log-Loss** (Cross-Entropy). It strictly penalizes models that are highly confident but entirely incorrect.

### The Formulation
For a true label $y \in \{0, 1\}$ and a predicted probability $p = P(y=1)$, the piecewise loss is:

$$
\text{loss}(\mathbf{w}) = 
\begin{cases}
-\log(p), & y=1 \\
-\log(1 - p), & y=0
\end{cases}
$$

In a single combined equation across all $N$ data points:
$$ \text{LogLoss} = -\frac{1}{N} \sum_{i=1}^N \left[ y_i \cdot \log(p_i) + (1 - y_i) \cdot \log(1 - p_i) \right] $$


In [ ]:
# Visualizing the Exponential Penalty of Log Loss
p_range = np.linspace(0.001, 0.999, 100)
loss_y1 = -np.log(p_range)
loss_y0 = -np.log(1 - p_range)

plt.figure(figsize=(10, 5))
plt.plot(p_range, loss_y1, color='lime', linewidth=3, label='Penalty if True Class $y = 1$')
plt.plot(p_range, loss_y0, color='red', linewidth=3, label='Penalty if True Class $y = 0$')
plt.title("Log-Loss Cost Function (Exponential Penalties)")
plt.xlabel("Predicted Probability ($p$)")
plt.ylabel("Log-Loss Cost")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 4. In-Class Activity: Hours Studied vs. Pass/Fail
As requested by the class activity instructions: 
> *Take the #hours studied vs. pass-fail usecase as an example to implement statistical classification log-loss.*

We will build the models from scratch and then verify with `scikit-learn`.


In [ ]:
# 1. Dataset Initialization
hours_studied = np.array([0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3, 3.25, 3.5, 3.75, 4, 4.25, 4.5, 4.75, 5])
pass_fail     = np.array([0,   0,    0, 0,    0,   0,    0, 0,    1,   0,    1, 0,    1,   1,    1, 1,    1,   1,    1])

X_reshaped = hours_studied.reshape(-1, 1)

# 2. Manual Algorithm for Log-Loss
def calculate_log_loss(y_true, y_prob):
    eps = 1e-15 # Prevent log(0)
    y_prob = np.clip(y_prob, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_prob) + (1 - y_true) * np.log(1 - y_prob))

print("Dataset Loaded Successfully!")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score

# 3. Model Training using Scikit-Learn
clf = LogisticRegression(random_state=42)
clf.fit(X_reshaped, pass_fail)

# 4. Predictions
predictions = clf.predict(X_reshaped)
probabilities = clf.predict_proba(X_reshaped)[:, 1]

# 5. Evaluate
acc = accuracy_score(pass_fail, predictions)
loss_sklearn = log_loss(pass_fail, probabilities)
loss_manual = calculate_log_loss(pass_fail, probabilities)

print(f"Model Accuracy: {acc * 100:.2f}%")
print(f"Scikit-Learn Log Loss: {loss_sklearn:.4f}")
print(f"Manual Log Loss:       {loss_manual:.4f}")


### Final Model Probability Curve Plot
Let's visually review exactly what the Logistic Regression model learned.


In [ ]:
# Generate continuous line for smooth plotting
x_test = np.linspace(0, 6, 100).reshape(-1, 1)
y_prob_test = clf.predict_proba(x_test)[:, 1]

plt.figure(figsize=(10, 6))
plt.scatter(hours_studied, pass_fail, color='orange', s=80, label='Actual Students Data', zorder=5)
plt.plot(x_test, y_prob_test, color='cyan', linewidth=3, label='Learned Logistic Curve')

# Find 0.5 decision boundary
decision_boundary = -clf.intercept_[0] / clf.coef_[0][0]
plt.axvline(decision_boundary, color='yellow', linestyle='--', label=f'Decision Boundary ({decision_boundary:.2f} hrs)')

plt.title("Logistic Regression Fit: Hours Studied vs Pass/Fail")
plt.xlabel("Hours Studied")
plt.ylabel("Probability of Passing")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 5. Peer Review Preparation (Group 2 & Group 5)

### 🗣️ Our Talking Points (To present to Group 5)
1. **The Math Behind the Curve**: We showed how the logistic function $\sigma(z)$ bends the linear combination into a smooth S-curve bounded between 0 and 1. We visualized this theoretically and practically on our dataset.
2. **Exponential Penalty of incorrect Confidence**: Our log-loss graphs demonstrate that if a model is 99% confident but wrong (e.g., predicting 99% pass for someone who actually failed), the $- \log$ penalty approaches infinity. MSE does not have this powerful property.
3. **Manual vs Library Parity**: We successfully built a manual function for Cross-Entropy loss from scratch using `numpy` and proved it mathematically matches the robust output of the `scikit-learn` backend identically.

---

### 📝 Live Interactive Note Space
*(To be filled during our 10 AM Session with Group 5)*

**Questions for Group 5:**
* Q1:
* Q2:

**Takeaways from Group 5's Implementation:**
* Note 1:
* Note 2:
